# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api_reference/dataset/) library.

### Dataset Source
The dataset source is specified via a Croissant schema URL, ensuring FAIR-compliant metadata standardization and access.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset by `@id`
record_sets = dataset.record_sets
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '(no name)')})")

# For demonstration, choose the main patient-level table record set
# [Update this if you know a different relevant record set]
main_record_set_id = record_sets[0]['@id'] if record_sets else None

# Inspect the fields and columns of the main record set
if main_record_set_id:
    print(f"\nFields in Record Set '{main_record_set_id}':")
    for record_set in record_sets:
        if record_set['@id'] == main_record_set_id:
            fields = record_set.get('field', [])
            if isinstance(fields, dict):
                fields = [fields]
            for f in fields:
                print(f"- Field @id: {f['@id']}, name: {f.get('name', '(no name)')}, dataType: {f.get('dataType', '(unknown)')}")
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. All entities are referenced by their `@id`s.

In [ ]:
# Prepare a list of all record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    # The generator yields dicts, one per record
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set '{record_set_id}'.")

# Choose the main record set for further EDA
if main_record_set_id in dataframes:
    print(f"\nColumns in DataFrame for '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No DataFrame available for record set '{main_record_set_id}'.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps:
- Select a numeric field using its `@id`.
- Filter, normalize, and group data. All references are by `@id`.

> _**Note:** Update the `numeric_field_id` and `group_field_id` after inspecting the real fields above!_

In [ ]:
# Replace these IDs with those appropriate for your dataset after examining the fields above
numeric_field_id = None
group_field_id = None

# Example: pick the first numeric field available for demo
field_types = {}
for rs in record_sets:
    if rs['@id'] == main_record_set_id:
        for f in rs.get('field', []):
            field_types[f['@id']] = f.get('dataType', None)

# Find a numeric-type field (`Integer` or `Float`)
for fid, dtype in field_types.items():
    if dtype in ['Integer', 'Float', 'Number']:
        numeric_field_id = fid
        break
# Use another field for grouping (string/categorical)
for fid, dtype in field_types.items():
    if fid != numeric_field_id and dtype and dtype.lower()=='text':
        group_field_id = fid
        break

print(f"Numeric field selected (by @id): {numeric_field_id}")
print(f"Group-by field selected (by @id): {group_field_id}")

# Proceed if fields are identified and data exists
if main_record_set_id in dataframes and numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id]
    # Ensure numeric conversion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    print(f"\nSummary statistics for {numeric_field_id}:")
    print(df[numeric_field_id].describe())

    threshold = df[numeric_field_id].quantile(0.25)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.4f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nFirst 5 normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the selected field, if it exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Unable to perform EDA: missing appropriate numeric/group fields or data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field if available
if main_record_set_id in dataframes and numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group field
if (
    main_record_set_id in dataframes 
    and numeric_field_id and group_field_id
    and numeric_field_id in dataframes[main_record_set_id].columns 
    and group_field_id in dataframes[main_record_set_id].columns
):
    plt.figure(figsize=(12,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to access, explore, and visualize a Croissant-standardized biomedical dataset using the `mlcroissant` library. By working with entities referenced by their `@id`s, your workflows remain robust, interoperable, and reproducible. For further analysis or specific research questions, customize the record set and field selections in the EDA steps above.